In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import ruptures as rpt

In [ ]:
ads = pd.read_csv('./data/ads_for_change_det.csv')

### prepare data

In [ ]:
# take subset of any symbol of interest
symbol = 'SUZLON' # SUZLON TATAMOTORS SBIN
df = ads.loc[ads['symbol'] == symbol].copy()

# smoothen the price using moving average
df['13w_ma'] = df['price_in_usd_7d_avg_ind'].rolling(window=13, center=True).mean()
# differentiate the price curve by calculating the lead (skipping the denominator of differential as the interval period is 1 unit (1 week in this case))
df['lead'] = df['13w_ma'].diff()
# smoothen the differential using moving average for better results in change point detectiona algorithm
df['lead_ma'] = df['lead'].rolling(window=13, center=True).mean()

# remove first and last few rows where we don't have moving average due to inadequate window size
df_non_null = df.dropna().reset_index(drop=True)
df_non_null

### change detection algorithm

In [ ]:
# minimum change intercal is 13 units (13 weeks in this case)
# model and jump arguments finalised by trial and error
algo = rpt.Pelt(model="rbf", min_size=13, jump=3).fit(df_non_null['lead_ma'].values)
result = algo.predict(pen=1)
# remove the last element as it is not an actual change point
result.pop()
result

### visualise change points in price chart

In [ ]:
plt.figure(figsize=(4,2))

plt.subplot(1,1,1)
# plot average price in fade colour and smoothened price in dark colour
plt.plot(df_non_null['price_in_usd_7d_avg_ind'], alpha = 0.3)
plt.plot(df_non_null['13w_ma'])
# for each change_point, get the corresponding price and plot in blue vertical line 
for change_point, price in zip(result, df_non_null.loc[result,'price_in_usd_7d_avg_ind'].values) :
    plt.vlines(change_point, ymin=price-150, ymax=price+150)
# add symbol name for reference
plt.text(x=0, y=max(df_non_null['13w_ma'])*0.1, s=symbol)
# save space
plt.axis('off')
plt.tight_layout()
plt.savefig('./change_detection.png')